# Chapitre 6 — Understanding the Question Before Searching

> Le 2ᵉ pilier — l'étape la plus négligée de toute la chaîne RAG.

Idée centrale : **une question utilisateur n'est PAS une seule entrée** — c'est la source de
deux préparations distinctes pour deux étapes différentes du pipeline :

| | Retrieval | Generation |
|---|---|---|
| Job | trouver les passages les plus probables | lire / raisonner / répondre |
| Bon à | matching de similarité | distinguer, exclure, formater |
| Mauvais à | rejeter précisément | chercher dans un corpus |
| Veut | rewrites en vocabulaire doc, anchor keywords, hints | question originale, format, disambiguation |

Le code companion vit dans [src/question/](../src/question/). Toutes les fonctions du chapitre
sont importables directement, et chaque cellule ci-dessous montre l'appel + la sortie.

## 0. Setup

In [ ]:
from __future__ import annotations

import os, sys
from pathlib import Path

if sys.platform == "win32":
    os.environ.setdefault("PYTHONUTF8", "1")
    try:
        sys.stdout.reconfigure(encoding="utf-8", errors="replace")
    except AttributeError:
        pass

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.question._llm import llm_available
print(f"REPO_ROOT     : {REPO_ROOT}")
print(f"OPENAI_API_KEY: {'set' if llm_available() else 'NOT SET → fallbacks deterministes'}")

In [ ]:
# Quelques questions test qui couvrent les cas du chapitre
QUESTIONS = {
    "effective_date": "What's the effective date of this contract? It's usually on page 1, top-right block.",
    "liability_cap":  "Find the liability cap. It's in the section called 'Risks and Limitations', usually subsection 3 or 4.",
    "max_coverage":   "What's the maximum coverage amount? Don't confuse it with the deductible — they're often listed together.",
    "flooding":       "Has flooding ever been excluded? Look in the schedule of exclusions, usually a table near the end.",
    "policy_number":  "What's the policy number? It's usually a code starting with 'POL-' or 'RC-' in the header.",
    "art_l131":       "Does article L131-1 of the insurance code apply here?",
    "exit_early":     "what happens if we exit early?",
    "cap_ambiguous":  "What's the cap?",
    "compare_caps":   "Are the indemnification and liability caps consistent in this contract?",
    "non_compete":    "Does this contract include a non-compete clause, and if so, for how long?",
    "format_iso":     "What's the effective date? Format: YYYY-MM-DD.",
    "limit_dk":       "What is the value of d_k for the original model with h=8 heads, NOT the values used in the ablations? It's usually in the section about scaled dot-product attention.",
}

for k, v in QUESTIONS.items():
    print(f"  {k:18s}: {v}")

## 1. Une question prépare DEUX choses, pas une

Prenons : *"What's the maximum coverage amount? Don't confuse it with the deductible — they're often listed together."*

Ce qu'il y a dedans :

- **Retrieval intent** → trouver le montant de couverture maximum
- **Scope cue** → ce contrat (implicite)
- **Layout hint** → les deux concepts sont souvent côte à côte (utile pour retrieval)
- **Disambiguation** → ne pas confondre avec la franchise (utile pour génération uniquement)

On ne peut PAS demander au retriever de "ne pas confondre" — ça n'a aucun sens vectoriel.
C'est pourquoi on split en deux artefacts.

In [ ]:
from src.question.types import RetrievalQuery, GenerationBrief, ParsedQuestion

# Démo : construction manuelle pour illustrer la dichotomie
demo = ParsedQuestion(
    retrieval=RetrievalQuery(
        main_query="What's the maximum coverage amount?",
        rewrites=["limit of indemnity", "maximum amount payable", "policy limit"],
        layout_hint="table",
    ),
    generation=GenerationBrief(
        original_question="What's the maximum coverage amount? Don't confuse it with the deductible.",
        disambiguation="NOT the deductible — extract only the coverage maximum.",
        must_distinguish=["deductible"],
    ),
)
print(demo.model_dump_json(indent=2))

## 2.1 Spell correction

BM25 et le matching exact cassent sur les typos. Embeddings y sont robustes mais pas gratuits.
Une passe LLM brève (mise en cache en prod) suffit. Sans `OPENAI_API_KEY`, fallback no-op.

In [ ]:
from src.question.spell import correct_spelling

for noisy in [
    "wht is the efective date of ths contrct?",
    "can u find the liabiliity cap pls",
    "What is the policy number?",  # déjà propre → identique
]:
    print(f"in : {noisy}")
    print(f"out: {correct_spelling(noisy)}\n")

## 2.2 Bridge the vocabulary gap

Variante de HyDE : on ne génère pas une fausse réponse, on génère **le langage qui entoure la
réponse**. Le retriever embarque la question + ses 3-5 reformulations, déduplique les hits, ranke.

In [ ]:
from src.question.rewrite import rewrite_query

rewrites = rewrite_query("what happens if we exit early?", domain_hint="commercial contract")
print("Rewrites pour: 'what happens if we exit early?'")
for r in rewrites:
    print(f"  • {r}")

## 2.3 Anchor keywords

Codes / acronymes / identifiants : à router vers BM25 ou un index regex en **parallèle** de
l'embedding (hybrid retrieval, chap. 7).

In [ ]:
from src.question.keywords import extract_anchor_keywords

samples = [
    "Does article L131-1 of the insurance code apply here?",
    "Find the GDPR clause and the SLA for ISO-9001 compliance — POL-2024 reference",
    "What's the policy number? It's usually a code starting with 'POL-' or 'RC-' in the header.",
]
for q in samples:
    print(f"{q}\n  → {extract_anchor_keywords(q)}\n")

## 2.4 Scope & structural hints

*« page 1 »*, *« section X »*, *« table near the end »*, *« in the header »* sont des filtres
à appliquer **avant** retrieval — deux ordres de magnitude de bruit éliminés gratuitement.

In [ ]:
from src.question.scope import extract_hints

for k in ("effective_date", "liability_cap", "flooding", "policy_number"):
    q = QUESTIONS[k]
    h = extract_hints(q)
    print(f"{k}")
    print(f"  Q: {q}")
    print(f"  → {h.model_dump(exclude_none=True)}\n")

## 2.5 Préparation retrieval complète

Tous les helpers 2.1–2.4 convergent dans `prepare_retrieval_query`.

In [ ]:
from src.question.retrieval_prep import prepare_retrieval_query

rq = prepare_retrieval_query(QUESTIONS["max_coverage"], domain_hint="insurance policy")
print(rq.model_dump_json(indent=2, exclude_none=True))

## 3.2 Format constraint (→ generation)

Une contrainte de format n'a RIEN à voir avec retrieval. C'est une instruction au générateur.

In [ ]:
from src.question.format import extract_format_constraint

for q in [
    "What's the effective date? Format: YYYY-MM-DD.",
    "Give me the answer as valid JSON.",
    "How much was paid in euros?",
    "Summarize in one sentence.",
    "List all exclusions as bullets.",
    "What's the policy number?",  # pas de contrainte
]:
    print(f"  {q!r:70s} → {extract_format_constraint(q)}")

## 3.3 Disambiguation cues (→ generation, JAMAIS retrieval)

Quand l'utilisateur dit *"not the deductible"*, c'est une consigne pour le LLM, pas un filtre.
On extrait le pattern et on le route vers `GenerationBrief`.

In [ ]:
from src.question.disambiguation import extract_disambiguation

for q in [
    "What is the limit per claim, not the deductible?",
    "Find the coverage amount, don't confuse it with the premium.",
    "Show termination rights instead of the renewal clause.",
    "What's the policy number?",  # rien à désambiguïser
    "Quel est le plafond, pas la franchise ?",
]:
    instruction, distractors = extract_disambiguation(q)
    print(f"  Q: {q}")
    print(f"     distractors: {distractors}")
    print(f"     instruction: {instruction}\n")

## 4. Pourquoi les exclusions vont à GENERATION, jamais à RETRIEVAL

Démonstration mécanique : sur la phrase

> *« The limit per claim is €1,500,000, with a deductible of €1,000. »*

Si on filtre au retrieval toute ligne contenant "deductible", on supprime EXACTEMENT la ligne qui
contient la réponse.

In [ ]:
passages = [
    "The limit per claim is €1,500,000, with a deductible of €1,000.",
    "The aggregate annual limit is €5,000,000.",
    "Deductibles apply per claim and per category — see schedule.",
    "Premium and renewal terms are described in section 7.",
]

user_question = "What is the limit per claim, not the deductible?"
_, distractors = extract_disambiguation(user_question)
print(f"Distractor détecté: {distractors}\n")

# ❌ Approche naïve : on filtre les passages contenant 'deductible' au retrieval
filtered = [p for p in passages if not any(d in p.lower() for d in distractors)]
print("❌ Filtrage AU RETRIEVAL (ce qu'il NE faut PAS faire) :")
for p in filtered:
    print(f"   {p}")
print("   → la 1ère ligne (qui contient la réponse) a été supprimée.\n")

# ✅ Approche correcte : on retrieve large, on désambiguïse au prompt génération
print("✅ Filtrage AU GENERATION (correct) :")
print("   passages livrés au LLM :")
for p in passages:
    print(f"     • {p}")
print("   + instruction au LLM : 'extract limit, NOT deductible'")
print("   → le LLM lit la 1ère ligne et renvoie 1 500 000 €, ignore les 1 000 € de franchise.")

## 5. Décomposition (questions multi-parties)

Comparaison, conditionnel, agrégation : impossibles à satisfaire en UN passage. On découpe.

In [ ]:
from src.question.classify import classify_intent
from src.question.decompose import decompose_query

for k in ("effective_date", "compare_caps", "non_compete", "flooding"):
    q = QUESTIONS[k]
    intent = classify_intent(q)
    decomp = decompose_query(q) if intent in ("compare", "aggregate", "conditional") else None
    print(f"[{intent:11s}]  {q}")
    if decomp:
        for i, sq in enumerate(decomp.sub_questions):
            deps = f" depends_on={sq.depends_on}" if sq.depends_on else ""
            print(f"   {i}. ({sq.operation}){deps}  {sq.text}")
    print()

## 6. Demander une clarification

L'option la plus sous-utilisée du pipeline. Si la question contient un référent ambigu et qu'il
n'y a pas de contexte conversationnel, on demande au lieu de deviner.

In [ ]:
from src.question.clarify import needs_clarification, suggest_clarifications

for q in [
    "What's the cap?",
    "Show me the latest version.",
    "Compare with last year.",
    "What's the effective date of contract POL-2024-117?",  # pas ambigu
]:
    if needs_clarification(q):
        print(f"❓ {q}")
        for s in suggest_clarifications(q):
            print(f"     • {s}")
    else:
        print(f"✅ {q}  (pas besoin de clarifier)")
    print()

## 7. Pipeline complet `understand_question`

Orchestration conditionnelle :

1. spell correction systématique  
2. clarification SI référent flou + pas d'historique → arrêt anticipé  
3. décomposition SI intent ∈ {compare, aggregate, conditional}  
4. `prepare_retrieval_query` + `GenerationBrief` par sous-question

In [ ]:
from src.question.pipeline import understand_question

def show_plan(q: str, *, history=None, domain="insurance policy"):
    plan = understand_question(q, conversation_history=history, domain_hint=domain)
    print("─" * 78)
    print(f"Q : {q}")
    print(f"action : {plan['action']}")
    if plan["action"] == "clarify":
        print("options :")
        for opt in plan["options"]:
            print(f"  • {opt}")
        return
    print(f"intent : {plan['intent']}")
    for i, pq in enumerate(plan["parsed"]):
        print(f"\n  Sub-question #{i}")
        print(f"  RETRIEVAL : main={pq.retrieval.main_query!r}")
        print(f"              rewrites={pq.retrieval.rewrites[:3]}{'...' if len(pq.retrieval.rewrites) > 3 else ''}")
        print(f"              anchors={pq.retrieval.anchor_keywords}")
        print(f"              page={pq.retrieval.page_hint}, section={pq.retrieval.section_hint}, layout={pq.retrieval.layout_hint}")
        print(f"  GENERATION: original={pq.generation.original_question!r}")
        print(f"              format={pq.generation.format_constraint}")
        print(f"              disambig={pq.generation.disambiguation}")
        print(f"              distinguish_from={pq.generation.must_distinguish}")

show_plan(QUESTIONS["max_coverage"])

In [ ]:
show_plan(QUESTIONS["cap_ambiguous"])              # → clarify
show_plan(QUESTIONS["compare_caps"])               # → décomposition

In [ ]:
show_plan(QUESTIONS["art_l131"])                   # anchor keyword → BM25
show_plan(QUESTIONS["flooding"])                   # layout hint=table

## Exercice — décomposition manuelle puis automatique

Question :

> *"What is the value of d_k for the original model with h=8 heads, NOT the values used in the ablations? It's usually in the section about scaled dot-product attention."*

À toi : décompose-la **mentalement** dans `RetrievalQuery` / `GenerationBrief` (lesquels morceaux
vont où ?), puis compare avec ce que `understand_question` produit ci-dessous.

In [ ]:
show_plan(QUESTIONS["limit_dk"], domain="deep learning paper")

**Lecture attendue :**

- `retrieval.main_query` → la question nettoyée
- `retrieval.rewrites` → "key dimension d_k in scaled dot-product attention", "head dimensionality in the base Transformer", ...
- `retrieval.section_hint` → "scaled dot-product attention"
- `retrieval.anchor_keywords` → vide ou bruit (pas de code régulier ici)
- `generation.original_question` → la question complète, telle que posée
- `generation.disambiguation` → "NOT the ablation values"
- `generation.must_distinguish` → ["ablations"] (ou approchant)

Le coup gagnant : la `section_hint` permet au retriever d'aller direct à la section *Scaled
Dot-Product Attention* via la `toc_df` (chap. 5), et la `disambiguation` empêche le LLM de
renvoyer les valeurs d'ablation par erreur.

## 8. Auto-audit

Sur ton pipeline actuel :

1. Corriges-tu les typos avant retrieval ?
2. Embeds-tu la question brute, ou des rewrites en vocabulaire-doc ?
3. Laisses-tu les utilisateurs inclure des hints (page, section, table, image) — et les utilises-tu ?
4. Sépares-tu intent de retrieval et contraintes de sortie ?
5. Gères-tu les exclusions au retrieval (cassé) ou à génération (correct) ?
6. Détectes-tu quand une question demande plusieurs retrievals ?
7. Extrais-tu le scope et l'utilises-tu comme filtre métadonnée avant retrieval ?
8. Demandes-tu parfois clarification, ou tu fonces ?

Si la majorité des réponses est *non*, la qualité fuit ici, en amont du retrieval. Aucun upgrade
d'embedding ni aucun tuning de chunk size ne le rattrapera — la fix vit upstream.